In [ ]:
import pandas as pd
import numpy as np
import fasttext
from huggingface_hub import hf_hub_download
from sklearn.cluster import MiniBatchKMeans
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


# Load Persian FastText model
model_path = hf_hub_download(
    repo_id="facebook/fasttext-fa-vectors",
    filename="model.bin"
)

ft = fasttext.load_model(model_path)


def mean_embedding(text):
    if not isinstance(text, str):
        return np.zeros(ft.get_dimension())

    tokens = text.split()
    vectors = [ft.get_word_vector(token) for token in tokens]

    return np.mean(vectors, axis=0)


# Load logistics data
df = pd.read_excel("data/Year_403.xlsx")

# Generate embeddings
df["embedding"] = df["product name"].apply(mean_embedding)

X = np.stack(df["embedding"].values)

# Unsupervised product categorization
kmeans = MiniBatchKMeans(
    n_clusters=12,
    random_state=42,
    batch_size=10000
)

df["cluster"] = kmeans.fit_predict(X)

# Train classification model
X_train, X_test, y_train, y_test = train_test_split(
    X,
    df["cluster"],
    test_size=0.2,
    random_state=42
)

model = SVC(kernel="linear")

model.fit(X_train, y_train)

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print(f"Model Accuracy: {accuracy:.4f}")


# Example prediction
sample_product = "میلگرد فولادی"

sample_vector = mean_embedding(sample_product).reshape(1, -1)

predicted_cluster = model.predict(sample_vector)[0]

print(f"Product: {sample_product}")
print(f"Predicted Cluster: {predicted_cluster}")